In [1]:
from IPython.display import display
import ipywidgets as widgets

uploader = widgets.FileUpload()
display(uploader)

FileUpload(value=(), description='Upload')

In [2]:
import zipfile
import os

# Create directories for extraction
os.makedirs("data/paysim", exist_ok=True)
os.makedirs("data/ieee", exist_ok=True)

# Unzip PaySim dataset
with zipfile.ZipFile("archive (1).zip", 'r') as zip_ref:
    zip_ref.extractall("data/paysim")

# Unzip IEEE-CIS dataset
with zipfile.ZipFile("ieee-fraud-detection.zip", 'r') as zip_ref:
    zip_ref.extractall("data/ieee")

print("✅ Datasets extracted successfully!")

✅ Datasets extracted successfully!


In [3]:
import os

print("📂 PaySim Files:", os.listdir("data/paysim"))
print("📂 IEEE Files:", os.listdir("data/ieee"))

📂 PaySim Files: ['PS_20174392719_1491204439457_log.csv']
📂 IEEE Files: ['sample_submission.csv', 'test_identity.csv', 'test_transaction.csv', 'train_identity.csv', 'train_transaction.csv']


In [4]:
import pandas as pd

# Load PaySim dataset
paysim_path = "data/paysim/PS_20174392719_1491204439457_log.csv"
df_paysim = pd.read_csv(paysim_path)

print("✅ PaySim Shape:", df_paysim.shape)
df_paysim.head()

✅ PaySim Shape: (6362620, 11)


,step,type,amount,nameOrig,oldbalanceOrg,newbalanceOrig,nameDest,oldbalanceDest,newbalanceDest,isFraud,isFlaggedFraud
0,1,PAYMENT,9839.64,C1231006815,170136.0,160296.36,M1979787155,0.0,0.0,0,0
1,1,PAYMENT,1864.28,C1666544295,21249.0,19384.72,M2044282225,0.0,0.0,0,0
2,1,TRANSFER,181.00,C1305486145,181.0,0.00,C553264065,0.0,0.0,1,0
3,1,CASH_OUT,181.00,C840083671,181.0,0.00,C38997010,21182.0,0.0,1,0
4,1,PAYMENT,11668.14,C2048537720,41554.0,29885.86,M1230701703,0.0,0.0,0,0


In [5]:
# Load IEEE datasets
train_transaction = pd.read_csv("data/ieee/train_transaction.csv")
train_identity = pd.read_csv("data/ieee/train_identity.csv")

# Merge on TransactionID
df_ieee = pd.merge(train_transaction, train_identity,
                   on="TransactionID", how="left")

print("✅ IEEE Dataset Shape:", df_ieee.shape)
df_ieee.head()

✅ IEEE Dataset Shape: (590540, 434)


,TransactionID,isFraud,TransactionDT,TransactionAmt,ProductCD,card1,card2,card3,card4,card5,...,id_31,id_32,id_33,id_34,id_35,id_36,id_37,id_38,DeviceType,DeviceInfo
0,2987000,0,86400,68.5,W,13926,NaN,150.0,discover,142.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2987001,0,86401,29.0,W,2755,404.0,150.0,mastercard,102.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2987002,0,86469,59.0,W,4663,490.0,150.0,visa,166.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2987003,0,86499,50.0,W,18132,567.0,150.0,mastercard,117.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2987004,0,86506,50.0,H,4497,514.0,150.0,mastercard,102.0,...,samsung browser 6.2,32.0,2220x1080,match_status:2,T,F,T,T,mobile,SAMSUNG SM-G892A Build/NRD90M


In [6]:
df_paysim.rename(columns={
    "nameOrig": "sender_account",
    "nameDest": "receiver_account",
    "type": "transaction_type",
    "amount": "transaction_amount",
    "isFraud": "is_fraud"
}, inplace=True)

# Map transaction types to Indian payment systems
mapping = {
    "TRANSFER": "IMPS",
    "CASH_OUT": "UPI",
    "DEBIT": "NEFT",
    "PAYMENT": "UPI"
}
df_paysim["transaction_type"] = df_paysim["transaction_type"].map(mapping)

print("✅ Columns renamed successfully!")

✅ Columns renamed successfully!


In [7]:
# Remove missing values
df_paysim.dropna(inplace=True)

# Check class distribution
print("Fraud Distribution:")
print(df_paysim["is_fraud"].value_counts(normalize=True))

Fraud Distribution:
is_fraud
0    0.998345
1    0.001655
Name: proportion, dtype: float64


In [8]:
df_paysim.to_csv("paysim_processed.csv", index=False)
print("File saved successfully!")

File saved successfully!


In [9]:
import pandas as pd
import os
import shutil

# ✅ Step 1: Load dataset
df = pd.read_csv("paysim_processed.csv")

# ✅ Step 2: Sort data (change column if needed)
df_sorted = df.sort_values(by="step")  # or another column if your data differs

# ✅ Step 3: Create folder for splits
os.makedirs("data/splits", exist_ok=True)

# ✅ Step 4: Split into 6 parts (banks)
chunk_size = len(df_sorted) // 6

for i in range(6):
    start = i * chunk_size
    end = (i + 1) * chunk_size if i < 5 else len(df_sorted)

    split = df_sorted.iloc[start:end]

    file_path = f"data/splits/bank{i+1}.csv"
    split.to_csv(file_path, index=False)
    print(f"✅ Saved {file_path}")

# ✅ Step 5: Zip all splits
shutil.make_archive("bank_splits", 'zip', "data/splits")

print("🎉 All files created successfully!")
print("📁 ZIP file: bank_splits.zip")

✅ Saved data/splits/bank1.csv
✅ Saved data/splits/bank2.csv
✅ Saved data/splits/bank3.csv
✅ Saved data/splits/bank4.csv
✅ Saved data/splits/bank5.csv
✅ Saved data/splits/bank6.csv
🎉 All files created successfully!
📁 ZIP file: bank_splits.zip


In [10]:
import shutil

shutil.make_archive("bank_splits", 'zip', "data/splits")
print("✅ ZIP file created successfully!")

✅ ZIP file created successfully!


In [11]:
from IPython.display import display
import ipywidgets as widgets

uploader = widgets.FileUpload()
display(uploader)

FileUpload(value=(), description='Upload')

In [12]:
import zipfile
import os

os.makedirs("data/splits", exist_ok=True)

with zipfile.ZipFile("bank_splits.zip", 'r') as zip_ref:
    zip_ref.extractall("data/splits")

print("✅ Bank datasets extracted!")
print(os.listdir("data/splits"))

✅ Bank datasets extracted!
['bank1.csv', 'bank2.csv', 'bank3.csv', 'bank4.csv', 'bank5.csv', 'bank6.csv']


In [13]:
!pip install torch torchvision torchaudio
!pip install torch-geometric -q
!pip install scikit-learn pandas numpy

In [14]:
import pandas as pd
import numpy as np
import torch
import torch.nn.functional as F
from torch_geometric.data import Data
from torch_geometric.nn import GCNConv
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import accuracy_score, classification_report
import matplotlib.pyplot as plt
import networkx as nx

In [15]:
import zipfile
import os

zip_path = "bank_splits.zip"  # Ensure this file is uploaded to Colab
extract_path = "data/splits"

os.makedirs(extract_path, exist_ok=True)

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

print("✅ Extracted files:", os.listdir(extract_path))

✅ Extracted files: ['bank1.csv', 'bank2.csv', 'bank3.csv', 'bank4.csv', 'bank5.csv', 'bank6.csv']


In [16]:
import pandas as pd
import torch
from torch_geometric.data import Data
from sklearn.preprocessing import StandardScaler

def create_graph(csv_path):
    df = pd.read_csv(csv_path)

    # --- Ensure numeric + clean ---
    df["transaction_amount"] = pd.to_numeric(df["transaction_amount"], errors='coerce')
    df = df.dropna(subset=["transaction_amount"])

    # --- Normalize transaction amount ---
    scaler = StandardScaler()
    df["transaction_amount"] = scaler.fit_transform(df[["transaction_amount"]])

    # --- Node mapping ---
    accounts = pd.concat([df["sender_account"], df["receiver_account"]]).unique()
    account_to_idx = {acc: i for i, acc in enumerate(accounts)}

    # --- Node Features ---
    sent_stats = df.groupby("sender_account")["transaction_amount"].agg(
        ["count", "mean", "sum"]
    ).rename_axis("account").reset_index()

    received_stats = df.groupby("receiver_account")["transaction_amount"].agg(
        ["count", "mean", "sum"]
    ).rename_axis("account").reset_index()

    node_features = pd.DataFrame({"account": accounts})
    node_features = node_features.merge(sent_stats, on="account", how="left") \
                                 .merge(received_stats, on="account", how="left",
                                        suffixes=("_sent", "_received")) \
                                 .fillna(0)

    x = torch.tensor(
        node_features.drop(columns=["account"]).values.astype(float),
        dtype=torch.float
    )

    # --- Edge index ---
    edge_index = torch.tensor([
        [account_to_idx[s] for s in df["sender_account"]],
        [account_to_idx[d] for d in df["receiver_account"]]
    ], dtype=torch.long)

    # --- Edge attributes (SAFE VERSION) ---
    edge_cols = [
        "transaction_amount",
        "transaction_type_0",
        "transaction_type_1",
        "transaction_type_2"
    ]

    # Ensure all columns exist (prevents KeyError)
    for col in edge_cols:
        if col not in df.columns:
            df[col] = 0

    edge_features = df[edge_cols]

    edge_attr = torch.tensor(
        edge_features.values.astype(float),
        dtype=torch.float
    )

    # --- Labels ---
    y = torch.tensor(df["is_fraud"].values, dtype=torch.long)

    return Data(x=x, edge_index=edge_index, edge_attr=edge_attr, y=y)

In [17]:
import torch
import torch.nn.functional as F
from torch_geometric.nn import GCNConv

class GNNFraud(torch.nn.Module):
    def __init__(self, input_dim, edge_dim, hidden_dim=64):
        super().__init__()
        self.conv1 = GCNConv(input_dim, hidden_dim)
        self.conv2 = GCNConv(hidden_dim, hidden_dim)

        # Edge-level classifier
        self.edge_mlp = torch.nn.Sequential(
            torch.nn.Linear(hidden_dim * 2 + edge_dim, hidden_dim),
            torch.nn.ReLU(),
            torch.nn.Linear(hidden_dim, 2)
        )

    def forward(self, data):
        x, edge_index, edge_attr = data.x, data.edge_index, data.edge_attr

        # Node embeddings
        x = F.relu(self.conv1(x, edge_index))
        x = F.relu(self.conv2(x, edge_index))

        # Edge embeddings
        src, dst = edge_index
        edge_emb = torch.cat([x[src], x[dst], edge_attr], dim=1)

        return self.edge_mlp(edge_emb)

In [19]:
import pandas as pd
import torch

# Convert amount
df["transaction_amount"] = pd.to_numeric(df["transaction_amount"], errors='coerce')
df = df.dropna(subset=["transaction_amount"])

# 🔥 HANDLE BOTH CASES
if "transaction_type" in df.columns:
    # Case 1: raw column → convert
    df = pd.get_dummies(df, columns=["transaction_type"])

# Now select all type columns automatically
type_cols = [col for col in df.columns if col.startswith("transaction_type")]

# Combine features
edge_features = df[type_cols].copy()
edge_features["transaction_amount"] = df["transaction_amount"]

# Convert to tensor
edge_attr = torch.tensor(
    edge_features.values.astype(float),
    dtype=torch.float
)

In [21]:
import torch
import pandas as pd

# --- Ensure amount is numeric ---
df["transaction_amount"] = pd.to_numeric(df["transaction_amount"], errors='coerce')
df = df.dropna(subset=["transaction_amount"])

# --- 🔍 DEBUG: check columns ---
print("Columns:", df.columns.tolist())

# --- HANDLE BOTH CASES ---
if any(col.startswith("transaction_type_") for col in df.columns):
    # ✅ Already one-hot encoded
    type_cols = [col for col in df.columns if col.startswith("transaction_type_")]
else:
    # ✅ Raw column → convert
    if "transaction_type" not in df.columns:
        raise ValueError("No transaction_type column found in dataset")

    df = pd.get_dummies(df, columns=["transaction_type"])
    type_cols = [col for col in df.columns if col.startswith("transaction_type_")]

# --- Build features ---
edge_features = df[type_cols].copy()
edge_features["transaction_amount"] = df["transaction_amount"]

# --- DEBUG ---
print(edge_features.dtypes)

# --- Convert to tensor ---
edge_attr = torch.tensor(
    edge_features.values.astype(float),
    dtype=torch.float
)

Columns: ['step', 'transaction_amount', 'sender_account', 'oldbalanceOrg', 'newbalanceOrig', 'receiver_account', 'oldbalanceDest', 'newbalanceDest', 'is_fraud', 'isFlaggedFraud', 'transaction_type_IMPS', 'transaction_type_NEFT', 'transaction_type_UPI']
transaction_type_IMPS       bool
transaction_type_NEFT       bool
transaction_type_UPI        bool
transaction_amount       float64
dtype: object


In [22]:
data = create_graph("data/splits/bank1.csv")
print(data)

Data(x=[1271195, 6], edge_index=[2, 827222], edge_attr=[827222, 4], y=[827222])


In [23]:
from sklearn.model_selection import train_test_split
import numpy as np
import torch

indices = np.arange(len(data.y))

train_idx, test_idx = train_test_split(
    indices,
    test_size=0.2,
    stratify=data.y.cpu().numpy(),
    random_state=42
)

train_mask = torch.zeros(len(data.y), dtype=torch.bool)
test_mask = torch.zeros(len(data.y), dtype=torch.bool)

train_mask[train_idx] = True
test_mask[test_idx] = True

data.train_mask = train_mask
data.test_mask = test_mask

In [29]:
model = GNNFraud(
    input_dim=data.x.shape[1],
    edge_dim=data.edge_attr.shape[1]
)

In [30]:
from sklearn.utils.class_weight import compute_class_weight
import numpy as np
import torch

weights = compute_class_weight(
    class_weight='balanced',
    classes=np.array([0, 1]),
    y=data.y.numpy()
)

# Emphasize the fraud class more
weights = torch.tensor([weights[0], weights[1] * 1.5], dtype=torch.float)
criterion = torch.nn.CrossEntropyLoss(weight=weights)

print("Class Weights:", weights)

Class Weights: tensor([5.0070e-01, 5.3856e+02])


In [31]:
from sklearn.utils import resample
import pandas as pd

df = pd.read_csv("data/splits/bank1.csv")

fraud = df[df.is_fraud == 1]
non_fraud = df[df.is_fraud == 0]

# Oversample fraud cases to 5% of non-fraud
fraud_upsampled = resample(
    fraud,
    replace=True,
    n_samples=int(len(non_fraud) * 0.05),
    random_state=42
)

df_balanced = pd.concat([non_fraud, fraud_upsampled])
df_balanced.to_csv("data/splits/bank1_balanced.csv", index=False)

In [35]:
model.eval()

with torch.no_grad():
    logits = model(data)

    probs = torch.softmax(logits[data.test_mask], dim=1)[:, 1].cpu().numpy()
    y_true = data.y[data.test_mask].cpu().numpy()

In [36]:
from sklearn.metrics import precision_recall_curve, classification_report
import numpy as np

precision, recall, thresholds = precision_recall_curve(y_true, probs)
f1_scores = 2 * (precision * recall) / (precision + recall + 1e-8)
best_threshold = thresholds[np.argmax(f1_scores)]

print("Best Threshold:", best_threshold)

# Apply the optimal threshold
preds_optimal = (probs >= best_threshold).astype(int)
print(classification_report(y_true, preds_optimal))

Best Threshold: 0.17444688
              precision    recall  f1-score   support

           0       1.00      0.00      0.00    165215
           1       0.00      1.00      0.00       230

    accuracy                           0.00    165445
   macro avg       0.50      0.50      0.00    165445
weighted avg       1.00      0.00      0.00    165445



In [37]:
import torch
torch.save(model.state_dict(), "bank1_gnn_model.pt")
print("✅ Model saved successfully!")

✅ Model saved successfully!


In [39]:
for i in range(1, 7):
    print(f"\n🚀 Training model for Bank {i}")
    data = create_graph(f"data/splits/bank{i}.csv")

    # Train/test split
    indices = np.arange(len(data.y))
    train_idx, test_idx = train_test_split(
        indices, test_size=0.2, stratify=data.y.numpy(), random_state=42
    )

    train_mask = torch.zeros(len(data.y), dtype=torch.bool)
    test_mask = torch.zeros(len(data.y), dtype=torch.bool)
    train_mask[train_idx] = True
    test_mask[test_idx] = True

    model = GNNFraud(
        input_dim=data.x.shape[1],
        edge_dim=data.edge_attr.shape[1]
    )

    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
    criterion = torch.nn.CrossEntropyLoss(weight=weights)

    for epoch in range(20):
        model.train()
        optimizer.zero_grad()
        out = model(data)
        loss = criterion(out[train_mask], data.y[train_mask])
        loss.backward()
        optimizer.step()

    torch.save(model.state_dict(), f"bank{i}_gnn_model.pt")
    print(f"✅ Saved bank{i}_gnn_model.pt")


🚀 Training model for Bank 1
✅ Saved bank1_gnn_model.pt

🚀 Training model for Bank 2
✅ Saved bank2_gnn_model.pt

🚀 Training model for Bank 3
✅ Saved bank3_gnn_model.pt

🚀 Training model for Bank 4
✅ Saved bank4_gnn_model.pt

🚀 Training model for Bank 5
✅ Saved bank5_gnn_model.pt

🚀 Training model for Bank 6
✅ Saved bank6_gnn_model.pt
